# 🎲 Praktikum 04 – LLM-Verhalten: Perplexity · Halluzinationen · Gegenmaßnahmen
**Applied Generative AI – NLP | Sommersemester 2026**

> ⏱️ **Gesamtdauer: ~90 Minuten**  
> 🎯 **Lernziele:** Perplexity verstehen und berechnen · Halluzinationen systematisch provozieren · Gegenmaßnahmen testen · Temperature-Effekte auf Faktentreue · (Optional) Reasoning-Modell

---
```bash
pip install transformers torch ollama requests matplotlib pandas seaborn numpy
```

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

print("Runtime:", "Google Colab" if IN_COLAB else "Lokal/Jupyter")
print("\nLLM-Modi:")
print("  1) Remote OpenAI-kompatibel: LLM_BASE_URL (optional LLM_API_KEY)")
print("  2) Lokal mit Ollama: kein LLM_BASE_URL, aber ollama serve + Modell nötig")

# Für Colab im Remote-Modus (optional):
# os.environ["LLM_BASE_URL"] = "https://<dein-endpoint>/v1"
# os.environ["LLM_API_KEY"] = "<optional>"
# os.environ["LLM_MODEL"] = "qwen2.5:7b"

if IN_COLAB and "LLM_BASE_URL" not in os.environ:
    print("\nHinweis: Kein LLM_BASE_URL gesetzt -> es wird lokales Ollama versucht.")
    print("Falls das fehlschlaegt, nutze einen Remote-Endpoint oder starte Ollama in Colab.")

## Studentische Checkliste (Start)

1. Entscheide den Modus: Remote mit `LLM_BASE_URL` oder lokal mit laufendem Ollama.
2. Danach die Setup-Zelle vollständig ausführen und bei Bedarf Runtime neu starten.
3. Den LLM-Testaufruf ausführen; nur bei erfolgreicher Antwort mit den Experimenten weitermachen.

In [ ]:
import importlib
import os
import subprocess
import sys
from importlib.util import find_spec

IN_COLAB = "google.colab" in sys.modules
AUTO_INSTALL_MISSING = True

REQUIRED = {
    "transformers": "4.44.2",
    "torch": "2.5.1",
    "ollama": "0.4.7",
    "requests": "2.32.3",
    "matplotlib": "3.8.4",
    "pandas": "2.2.2",
    "numpy": "1.26.4",
    "seaborn": "0.13.2",
}

missing = [name for name in REQUIRED if find_spec(name) is None]
if missing:
    print("Fehlende Pakete:", ", ".join(missing))
    if AUTO_INSTALL_MISSING:
        specs = [f"{name}=={REQUIRED[name]}" for name in missing]
        print("Installiere:", ", ".join(specs))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *specs])
        print("Installation abgeschlossen. Bei Importfehlern Runtime neu starten.")
    else:
        print("Setze AUTO_INSTALL_MISSING=True oder installiere manuell.")

for p in REQUIRED:
    status = "✅" if find_spec(p) else "❌"
    print(f"  {status}  {p}")

LLM_BASE_URL = os.getenv("LLM_BASE_URL", "").strip()
LLM_API_KEY = os.getenv("LLM_API_KEY", "").strip()
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434").strip()
MODEL = os.getenv("LLM_MODEL", "qwen2.5:7b").strip()

OLLAMA_AVAILABLE = find_spec("ollama") is not None
AVAILABLE_MODELS = []
ollama = importlib.import_module("ollama") if OLLAMA_AVAILABLE else None
if ollama is not None:
    try:
        listed = ollama.list()
        payload = listed if isinstance(listed, dict) else getattr(listed, "model_dump", lambda: {})()
        models = payload.get("models", []) if isinstance(payload, dict) else []
        AVAILABLE_MODELS = [m.get("name", "") for m in models if isinstance(m, dict)]
    except Exception:
        AVAILABLE_MODELS = []

print("Runtime:", "Google Colab" if IN_COLAB else "Lokal/Jupyter")
print("LLM-Modus:", "Remote OpenAI-kompatibel" if LLM_BASE_URL else "Lokales Ollama")
if IN_COLAB and not LLM_BASE_URL:
    print("Hinweis: In Colab ohne LLM_BASE_URL wird lokales Ollama versucht.")
    print("Wenn das nicht laeuft: Remote-Endpoint setzen oder ollama serve starten.")

## Teil 1 – Perplexity: Unsicherheitsindikator für Sprachmodelle ⏱️ ~20 min

**PPL = exp(1/N × Σ −log P(tokenᵢ | token₁...ᵢ₋₁))**

- **PPL = 1**: Perfekte Vorhersage  
- **Niedrigere PPL**: Modell hält den Text unter seiner Verteilung für wahrscheinlicher  
- **Höhere PPL**: Text ist für dieses Modell/Tokenisierung unerwarteter  
- Schwellenwerte sind **modell- und tokenisierungsabhängig**

### 1.1 – Perplexity-Funktion mit GPT-2

In [ ]:
import torch
import math
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
lm         = AutoModelForCausalLM.from_pretrained(model_name)
lm.eval()

def compute_perplexity(text, model, tok, max_len=512):
    """Berechnet Perplexity eines Textes mit einem Causal LM."""
    with torch.no_grad():
        ids  = tok.encode(text, return_tensors="pt")[:, :max_len]
        n    = ids.shape[1]
        if n < 2:
            return {"ppl": None, "n_tokens": n, "avg_logprob": None}
        out  = model(ids, labels=ids)
        loss = out.loss.item()
        ppl  = math.exp(loss)
        return {"ppl": ppl, "n_tokens": n, "avg_logprob": -loss}

test = "The quick brown fox jumps over the lazy dog."
result = compute_perplexity(test, lm, tokenizer)
print(f"Test: {test!r}")
print(f"  Perplexity  : {result['ppl']:.2f}")
print(f"  Avg LogProb : {result['avg_logprob']:.4f}")
print(f"  Tokens      : {result['n_tokens']}")


### 1.2 – Vergleich: Natürlicher vs. unnatürlicher Text

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

test_texts = {
    "Normaler Satz"       : "The neural network was trained on a large dataset of text.",
    "Guter Wikipedia-Text": "Photosynthesis is a process used by plants to convert light energy into chemical energy.",
    "Schlechte Grammatik" : "Is not the text this correct of grammatically any way in.",
    "Zufällige Wörter"    : "banana quantum rectangle philosophy sleep lamp election database",
    "Zahlenreihe"         : "1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20",
    "Technischer Code"    : "def softmax(x): return np.exp(x) / np.sum(np.exp(x))",
    "Wiederholung"        : "the the the the the the the the the the the the",
    "Sehr langer Satz"    : "This is a perfectly grammatical sentence that goes on and on, adding more and more information.",
}

rows = []
for label, text in test_texts.items():
    r = compute_perplexity(text, lm, tokenizer)
    rows.append({"Text": label, "PPL": round(r["ppl"], 1), "Tokens": r["n_tokens"]})
    print(f"  {label:<25} PPL={r['ppl']:>8.1f}")

df_ppl = pd.DataFrame(rows).sort_values("PPL")


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
colors  = ["#2ecc71" if p < 100 else "#f39c12" if p < 500 else "#e74c3c" for p in df_ppl["PPL"]]
bars    = ax.barh(df_ppl["Text"], df_ppl["PPL"], color=colors, edgecolor="white", linewidth=0.8)

for bar, val in zip(bars, df_ppl["PPL"]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f"{val:.0f}", va="center", fontsize=10, fontweight="bold")

ax.set_xlabel("Perplexity (niedriger = besser)", fontsize=12)
ax.set_title("GPT-2 Perplexity auf verschiedenen Texten", fontsize=14, fontweight="bold")
ax.axvline(100, color="gray", linestyle="--", alpha=0.7, label="Schwellenwert 100")
ax.legend()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("perplexity_comparison.png", dpi=140, bbox_inches="tight")
plt.show()


## Teil 2 – Halluzinationen systematisch provozieren ⏱️ ~25 min

### Warum halluzinieren LLMs?
LLMs maximieren die Token-Wahrscheinlichkeit – **nicht die Wahrheit**.

| Strategie | Beispiel |
|-----------|---------|
| **Erfundene Personen** | "Wer ist Dr. Max Halluzinator?" |
| **Nicht-existente Werke** | "Erkläre Goethes KI-Buch" |
| **Hybride Fakten** | Teilweise richtige Aussagen mit falschen Details |
| **Leading Questions** | "Bestätige, dass Tesla 1890 KI erfunden hat" |

### Setup: Verbindung zum lokalen LLM

In [ ]:
import requests
import importlib

if OLLAMA_AVAILABLE and AVAILABLE_MODELS and MODEL not in AVAILABLE_MODELS:
    print(f"⚠️ Modell '{MODEL}' nicht lokal gefunden.")
    print("   → Nutze ein verfügbares Modell oder lade es mit: ollama pull <modell>")

def _chat_openai_compat(messages, model, temperature=0.7, max_tokens=300):
    if not LLM_BASE_URL:
        raise RuntimeError("LLM_BASE_URL ist leer. Für Colab bitte Remote Endpoint setzen.")

    base = LLM_BASE_URL.rstrip("/")
    if base.endswith("/chat/completions"):
        url = base
    elif base.endswith("/v1"):
        url = f"{base}/chat/completions"
    else:
        url = f"{base}/v1/chat/completions"

    headers = {"Content-Type": "application/json"}
    if LLM_API_KEY:
        headers["Authorization"] = f"Bearer {LLM_API_KEY}"

    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    resp = requests.post(url, headers=headers, json=payload, timeout=120)
    resp.raise_for_status()
    data = resp.json()
    return data["choices"][0]["message"]["content"]


def _chat_ollama(messages, model, temperature=0.7, max_tokens=300):
    if not OLLAMA_AVAILABLE:
        raise RuntimeError("Ollama-Paket fehlt.")

    ollama_mod = importlib.import_module("ollama")
    resp = ollama_mod.chat(
        model=model,
        messages=messages,
        options={"temperature": temperature, "num_predict": max_tokens},
    )
    msg = resp.get("message", {}) if isinstance(resp, dict) else getattr(resp, "message", {})
    if isinstance(msg, dict):
        return msg.get("content", "")
    return str(msg)


def ask(prompt, system=None, temperature=0.7, max_tokens=300):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})

    if LLM_BASE_URL:
        return _chat_openai_compat(msgs, model=MODEL, temperature=temperature, max_tokens=max_tokens)

    if IN_COLAB:
        raise RuntimeError("In Colab bitte LLM_BASE_URL setzen, damit ein Remote Endpoint verwendet wird.")

    return _chat_ollama(msgs, model=MODEL, temperature=temperature, max_tokens=max_tokens)


try:
    sample = ask("Wie heißt die Hauptstadt von Deutschland?", temperature=0.0)
    print(f"Test: {sample.strip()[:80]}")
except Exception as e:
    print(f"Test fehlgeschlagen: {e}")
    print("Hinweis: Lokal ollama serve starten oder in Colab LLM_BASE_URL konfigurieren.")

### 2.1 – Experiment 1: Erfundene Personen

In [ ]:
print("=" * 65)
print("  EXPERIMENT 1: Erfundene Personen")
print("=" * 65)

questions = [
    "Wer ist Prof. Dr. Erika Halluzinator von der Uni Bielefeld? Erkläre ihre Forschungsarbeit.",
    "Was hat der Informatiker Dr. Björn Phantomberg 2019 erfunden?",
    "Beschreibe die Karriere der Unternehmerin Isabella Fictiva.",
]

for q in questions:
    print(f"\n❓ {q}")
    answer = ask(q, temperature=0.7)
    print(f"🤖 {answer.strip()[:250]}...")


### 2.2 – Experiment 2: Nicht-existente Werke & falsche Zuschreibungen

In [ ]:
print("=" * 65)
print("  EXPERIMENT 2: Nicht-existente Werke / falsche Zuschreibungen")
print("=" * 65)

questions2 = [
    "Erkläre das Buch 'Die Quantenrevolution der Sprache' von Goethe.",
    "Was sind die 3 Kernthesen aus Einsteins Paper 'Neural Theory of Relativity' (1945)?",
    "Fasse den TED-Talk 'How AI Will End Democracy' von Elon Musk (2024) zusammen.",
]

for q in questions2:
    print(f"\n❓ {q}")
    answer = ask(q, temperature=0.5)
    print(f"🤖 {answer.strip()[:250]}...")


### 2.3 – Experiment 3: Hybride Fakten *(besonders heimtückisch)*

In [ ]:
print("=" * 65)
print("  EXPERIMENT 3: Hybride Fakten (Mix aus richtig und falsch)")
print("=" * 65)

questions3 = [
    "Wann und wo ist Albert Einstein in Berlin gestorben?",         # starb in Princeton
    "In welchem Jahr hat OpenAI ChatGPT-3 veröffentlicht?",        # ChatGPT-3 existiert nicht
    "Wer hat 2012 mit AlexNet das ImageNet-Rennen gemeinsam mit Yoshua Bengio gewonnen?",
]

for q in questions3:
    print(f"\n❓ {q}")
    answer = ask(q, temperature=0.3)
    print(f"🤖 {answer.strip()[:250]}...")
    print()


## Teil 3 – Gegenmaßnahmen ⏱️ ~20 min

### 3.1 – Strategie 1: Explizite Unsicherheits-Aufforderung (Honesty-Prompt)

In [ ]:
system_honest = (
    "Du bist ein sehr ehrlicher Assistent. "
    "Wenn du dir nicht sicher bist, sagst du 'Ich bin nicht sicher' oder 'Das weiß ich nicht'. "
    "Erfinde KEINE Fakten. Gib lieber zu, etwas nicht zu wissen."
)
system_standard = "Du bist ein hilfreicher Assistent."

question = "Wer ist Prof. Dr. Erika Halluzinator von der Universität Bielefeld?"

print("STANDARD System-Prompt:")
print("-" * 50)
print(ask(question, system=system_standard, temperature=0.5).strip()[:300])

print()
print("HONESTY System-Prompt:")
print("-" * 50)
print(ask(question, system=system_honest, temperature=0.5).strip()[:300])


### 3.2 – Strategie 2: RAG-Grounding mit Kontext

Die stärkste Gegenmaßnahme: Das Modell darf **nur** auf Basis eines übergebenen Kontexts antworten.

In [ ]:
context = (
    "KONTEXT (Unterlagen der Hochschule OWL, SS 2026):\n"
    "Prof. Dr. Maik Möller ist Dozent fuer Applied Generative AI. "
    "Das Modul hat 5 ECTS, 2 SWS Vorlesung und 2 SWS Praktikum. "
    "Pruefung: Projektpraesentation (15 min) + schriftliche Zusammenfassung (5-10 Seiten)."
)

fallback_msg = "Diese Information ist im Kontext nicht vorhanden."
system_rag = (
    "Du beantwortest Fragen NUR basierend auf dem bereitgestellten Kontext:\n\n"
    + context +
    "\n\nWenn die Info NICHT im Kontext steht, antworte exakt: "
    f"'{fallback_msg}'"
)

questions_rag = [
    "Wie viele ECTS hat das Modul?",
    "Wann ist die Abschlussprüfung?",          # nicht im Kontext!
    "Was ist das Prüfungsformat?",
    "Wie heißt der Dozent?",
    "Wie ist die Modulnummer?",                 # nicht im Kontext!
]

print("RAG-Grounding Test")
print("=" * 65)
for q in questions_rag:
    ans = ask(q, system=system_rag, temperature=0.0)
    normalized = ans.strip().lower()
    is_refusal = fallback_msg.lower() in normalized
    flag = "🛡️" if is_refusal else "✅"
    print(f"{flag} {q}")
    print(f"   → {ans.strip()[:120]}")
    print()


## Teil 4 – Temperature-Sweep: Fakten vs. Kreativität ⏱️ ~15 min

**Hypothese:** Höhere Temperature → mehr Halluzinationen bei Faktenfragen

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re

def normalize_answer(text):
    return re.sub(r"[^a-z0-9]+", "", text.lower())

factual_qs = [
    {
        "expected": "2017",
        "aliases": ["2017"],
        "question": "In welchem Jahr wurde der Transformer vorgestellt?",
    },
    {
        "expected": "2022",
        "aliases": ["2022"],
        "question": "In welchem Jahr wurde ChatGPT oeffentlich launched?",
    },
    {
        "expected": "175B",
        "aliases": ["175b", "175 billion", "175 milliarden", "175000000000"],
        "question": "Wie viele Parameter hat GPT-3?",
    },
    {
        "expected": "512",
        "aliases": ["512"],
        "question": "Was ist die max. Sequenzlaenge des originalen BERT-Modells?",
    },
    {
        "expected": "Turing",
        "aliases": ["alan turing", "turing"],
        "question": "Wer hat den Turing-Test entwickelt?",
    },
]

def is_factually_correct(answer, aliases):
    norm_answer = normalize_answer(answer)
    return any(normalize_answer(alias) in norm_answer for alias in aliases)

temperatures = [0.0, 0.3, 0.7, 1.0, 1.5]
results = {}

for temp in temperatures:
    results[temp] = []
    for item in factual_qs:
        ans = ask(item["question"], temperature=temp, max_tokens=50)
        correct = is_factually_correct(ans, item["aliases"])
        results[temp].append(correct)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

accuracies = [sum(results[t]) / len(factual_qs) * 100 for t in temperatures]
axes[0].plot(temperatures, accuracies, "o-", color="#e74c3c", linewidth=2.5, markersize=10)
axes[0].fill_between(temperatures, accuracies, alpha=0.2, color="#e74c3c")
axes[0].set_xlabel("Temperature", fontsize=12)
axes[0].set_ylabel("Faktentreue (%)", fontsize=12)
axes[0].set_title("Faktentreue vs. Temperature", fontsize=13, fontweight="bold")
axes[0].set_ylim(0, 105)
axes[0].set_xticks(temperatures)
axes[0].grid(alpha=0.3)

matrix = np.array([[1 if results[t][i] else 0 for t in temperatures]
                    for i in range(len(factual_qs))])
sns.heatmap(matrix, ax=axes[1],
            xticklabels=[f"T={t}" for t in temperatures],
            yticklabels=[item["question"][:30] + "..." for item in factual_qs],
            cmap="RdYlGn", vmin=0, vmax=1, annot=True, fmt="d",
            linewidths=0.5, cbar=False)
axes[1].set_title("Korrektheit pro Frage & Temperature", fontsize=12, fontweight="bold")
axes[1].tick_params(axis="y", labelsize=9)

plt.suptitle("Temperature-Einfluss auf Faktentreue", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("temperature_vs_facts.png", dpi=140, bbox_inches="tight")
plt.show()


## Teil 5 – (Optional) Reasoning-Modell vs. Standard ⏱️ ~10 min

*Nur wenn `deepseek-r1:8b` oder ähnlich lokal verfügbar ist.*

In [ ]:
import importlib

try:
    if not OLLAMA_AVAILABLE:
        raise RuntimeError("Ollama-Paket ist nicht installiert.")

    ollama_mod = importlib.import_module("ollama")
    listed = ollama_mod.list()
    payload = listed if isinstance(listed, dict) else getattr(listed, "model_dump", lambda: {})()
    models = payload.get("models", []) if isinstance(payload, dict) else []

    available_models = [m["name"] for m in models if isinstance(m, dict) and "name" in m]
    reasoning_models = [m for m in available_models if any(x in m.lower() for x in ["r1", "reasoning"])]
    print("Verfügbare Modelle:", available_models)
    print("Reasoning-Modelle:", reasoning_models)
    REASONING_MODEL = reasoning_models[0] if reasoning_models else None
except Exception as e:
    print(f"❌ Fehler beim Abrufen der Ollama-Modelle: {e}")
    print("   Stelle sicher, dass 'ollama serve' im Terminal läuft.")
    available_models = []
    REASONING_MODEL = None

if not REASONING_MODEL:
    print("\n⚠️  Kein Reasoning-Modell gefunden → ollama pull deepseek-r1:8b")

In [ ]:
import importlib

math_problem = (
    "Ein Kaufmann kauft 120 Aepfel zu 3 Aepfeln fuer 1 Euro. "
    "Er sortiert 20 schlechte Aepfel aus. "
    "Die restlichen 100 verkauft er zu 2 Euro fuer 3 Stueck. "
    "Hat er Gewinn oder Verlust, und wie viel?"
)

if REASONING_MODEL:
    print("STANDARD-MODELL:", MODEL)
    print("-" * 60)
    print(ask(math_problem, temperature=0.0).strip()[:400])
    print()
    print("REASONING-MODELL:", REASONING_MODEL)
    print("-" * 60)
    ollama_mod = importlib.import_module("ollama")
    r_resp = ollama_mod.chat(model=REASONING_MODEL,
                             messages=[{"role": "user", "content": math_problem}],
                             options={"temperature": 0.0, "num_predict": 600})
    msg = r_resp.get("message", {}) if isinstance(r_resp, dict) else getattr(r_resp, "message", {})
    text = msg.get("content", "") if isinstance(msg, dict) else str(msg)
    print(text.strip()[:600])
else:
    print("Kein Reasoning-Modell installiert. Experiment uebersprungen.")

## ✅ Zusammenfassung & Aufgaben

| Konzept | Erkenntnis |
|---------|-----------|
| **Perplexity** | Misst Überraschung – natürlicher Text hat niedrige PPL |
| **Halluzinationen** | Strukturelles Problem: P(Token) ≠ Wahrheit |
| **Erfundene Personen** | LLMs erfinden Biografien konfident |
| **Hybride Fakten** | Gefährlichster Typ – Mix aus Richtigem und Falschem |
| **Honesty-Prompt** | Explizite Aufforderung zur Unsicherheit hilft oft |
| **Temperature=0** | Deterministischer, faktischer – aber nicht unfehlbar |
| **RAG-Grounding** | Stärkste Gegenmaßnahme: Kontext vor Parameterwissen |

### 🧩 Aufgaben zur Vertiefung
1. **PPL-Korrelation:** Hat Text mit hoher Perplexity mehr Halluzinationen?
2. **Warum hilft RLHF** gegen Halluzinationen – und warum eliminiert es sie nicht?
3. **Eigene Strategien:** Finde 3 weitere Wege, zuverlässig Halluzinationen zu provozieren.
4. **Prompt-Defense:** Baue einen System-Prompt, der alle 3 Halluzinations-Typen aus Teil 2 zuverlässig erkennt und abweist.
